# 현재 구조: Express 추천 + Python 리뷰 분석 작업
이 노트북은 모의 데이터로 Express 추천 테스트와 Python 리뷰 저장 작업을 검증합니다.
실제 DB, Gemini, 별도 Python HTTP 서버에 연결하지 않습니다.


# 모의 데이터 개발용
각 코드 셀은 위 경로 설정 셀 외의 선행 셀 출력이나 저장 artifact에 의존하지 않습니다.
Express 테스트는 자체 모의 데이터와 임시 리뷰 파일을 생성·정리합니다.


In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
AI_ROOT = next(p for base in (cwd, *cwd.parents) for p in (base, base / 'ai-jobs')
               if (p / 'app/review_summary.py').is_file())
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))


In [ ]:
from tempfile import TemporaryDirectory
from dataclasses import replace
from app.review_summary import Review, SummaryService, MockReviewSummarizer
from app.repositories.reviews import MockReviewRepository
reviews = [Review(1, 990001, '평일에는 조용해요.'),
           Review(2, 990001, '주말에는 시끄러워요.'),
           Review(3, 990001, '비공개 의견', status='HIDDEN')]
with TemporaryDirectory() as folder:
    service = SummaryService(Path(folder) / 'summary.sqlite', MockReviewSummarizer())
    review_repo = MockReviewRepository(reviews)
    summary, changed = service.refresh(990001, review_repo.list_reviews(990001))
    assert changed and summary['review_count'] == 2
    print('주제별 의견:', summary['topics'])
    print('근거 원문:', summary['opinions'])
    assert summary['topics'][0]['mixed_opinions'] is True
    assert not service.refresh(990001, review_repo.list_reviews(990001))[1]
    reviews[0] = replace(reviews[0], content='좌석이 넓어요.')
    assert service.refresh(990001, MockReviewRepository(reviews).list_reviews(990001))[1]
    reviews[0] = replace(reviews[0], deleted_at='2026-11-09')
    summary, changed = service.refresh(990001, MockReviewRepository(reviews).list_reviews(990001))
    assert changed and summary['source_review_ids'] == [2]
    print(summary)
print('리뷰 조회/저장/재사용/수정/삭제 PASS / 임시 파일 삭제 완료')


### 구조화 요약 판정
- `opinions`: 원문 발췌, 리뷰 ID, 모의 주제/긍정·부정 분류
- `topics`: 분류별 근거 ID와 서로 다른 의견 존재 여부
- `mixed_opinions`: 의견 차이가 있다는 의미이며 사실 모순을 판정하지 않습니다.
- 날짜·상황을 원문 그대로 보존합니다. 대상 리뷰 수와 실제 발췌 수를 구분합니다.
- 현재 분류는 고정 모의 문장에만 적용됩니다. 일반 문장은 other/neutral이며 LLM 평가가 아닙니다.


## 리뷰 요약 저장 검증
Python 모의 리뷰 요약은 임시 SQLite에서 실행합니다. Express 추천·모의 리뷰 조회는 아래 Node 테스트로 검증합니다.


In [ ]:
import os, shutil, subprocess
npm = shutil.which('npm.cmd' if os.name == 'nt' else 'npm')
node = shutil.which('node')
if not npm or not node:
    raise RuntimeError('Node.js/npm을 설치하세요.')
subprocess.run([npm, 'run', 'build'], check=True, cwd=AI_ROOT.parent / 'backend', timeout=60)
subprocess.run([node, '--test', 'tests/recommendations.cjs'], check=True,
               cwd=AI_ROOT.parent / 'backend', timeout=60)


Express 테스트는 임시 리뷰 파일을 자체 생성·정리하므로 Python 내보내기가 필요 없습니다. 수동 리뷰 API 시연에만 `scripts/export_mock_review_summary.py`를 별도로 실행하세요. 운영 차단은 `NODE_ENV=production` 지정에 의존합니다.


## 리뷰 Agent 작업 (모의)
실제 API·DB 없이 분석 → 근거 검증 → 버전 확인 저장을 실행합니다. Express 파일 조회와 별개의 격리 실습입니다.


In [ ]:
from app.review_job import ReviewJobRunner, MockReviewModel
from app.repositories.analyses import MockAnalysisRepository, ReviewInput
analysis_repo = MockAnalysisRepository()
analysis_repo.replace_reviews(1, (ReviewInput(1, '평일에는 조용해요'), ReviewInput(2, '주말에는 시끄러워요')))
job_result = ReviewJobRunner(analysis_repo, MockReviewModel()).run(1)
assert job_result['status'] == 'READY'
print(job_result)
print(analysis_repo.current(1)['opinions'])
